# AI Code Review Agent


In [4]:
import sys

!{sys.executable} -m pip install -q langchain
!{sys.executable} -m pip install -q langchain-core
!{sys.executable} -m pip install -q langchain-groq
!{sys.executable} -m pip install -q streamlit
!{sys.executable} -m pip install -q python-dotenv

## Set Your Groq API Key


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


## Define the 4 Agent Tools


In [7]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def get_llm():
    return ChatGroq(
        groq_api_key=os.environ["GROQ_API_KEY"],
        model_name="llama-3.3-70b-versatile",
        temperature=0.1,
        max_tokens=2048
    )


#### TOOL 1: BUG DETECTOR

In [10]:

def detect_bugs(code, language="Python"):
    print("Tool 1: Bug Detection running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert code reviewer. Find ALL bugs in the code. "
         "For each bug: number, description, line, severity (CRITICAL/HIGH/MEDIUM/LOW), fix. "
         "If no bugs say: No bugs found."),
        ("human", "Find bugs in this {language} code:\n{language}\n{code}\n")
    ])
    return (prompt | get_llm() | StrOutputParser()).invoke({"language": language, "code": code})


#### TOOL 2: SECURITY CHECKER 

In [11]:

def check_security(code, language="Python"):
    print("Tool 2: Security Analysis running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a cybersecurity expert. Find ALL security issues: SQL injection, "
         "hardcoded passwords, missing validation, etc. "
         "For each: name, risk (CRITICAL/HIGH/MEDIUM/LOW), what attacker can do, how to fix. "
         "If no issues say: No security issues found."),
        ("human", "Security audit this {language} code:\n{language}\n{code}\n")
    ])
    return (prompt | get_llm() | StrOutputParser()).invoke({"language": language, "code": code})


#### TOOL 3: TEST GENERATOR 

In [12]:

def generate_tests(code, language="Python"):
    print("Tool 3: Unit Test Generation running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a senior software engineer. Write complete runnable unit tests. "
         "Use pytest for Python. Cover: normal cases, edge cases, error cases."),
        ("human", "Write unit tests for this {language} code:\n{language}\n{code}\n")
    ])
    return (prompt | get_llm() | StrOutputParser()).invoke({"language": language, "code": code})


#### TOOL 4: DOC GENERATOR

In [13]:

def generate_docs(code, language="Python"):
    print("Tool 4: Documentation Generation running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a technical writer. Write professional documentation including: "
         "overview, each function with parameters/returns/examples, dependencies, usage example."),
        ("human", "Document this {language} code:\n{language}\n{code}\n")
    ])
    return (prompt | get_llm() | StrOutputParser()).invoke({"language": language, "code": code})

print(" All 4 tools defined!")

 All 4 tools defined!


#### Run Agent on Sample Code


In [14]:
sample_code = """
import sqlite3

def get_user(username, password):
    conn   = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query  = "SELECT * FROM users WHERE username = '" + username + "'"
    cursor.execute(query)
    user   = cursor.fetchone()
    if user and user[2] == password:
        return user
    return None

def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    return total / len(numbers)

def save_config():
    api_key = "sk-secret123456789"
    with open('config.txt', 'w') as f:
        f.write(api_key)
"""

LANGUAGE = "Python"

print("AI Code Review Agent Starting...")
print(f"Language: {LANGUAGE} | Lines: {len(sample_code.strip().splitlines())}\n")

bug_result  = detect_bugs(sample_code, LANGUAGE)
sec_result  = check_security(sample_code, LANGUAGE)
test_result = generate_tests(sample_code, LANGUAGE)
docs_result = generate_docs(sample_code, LANGUAGE)

print("\n" + "="*50)
print("ALL 4 TOOLS COMPLETE!")
print("="*50)

AI Code Review Agent Starting...
Language: Python | Lines: 22

Tool 1: Bug Detection running...
Tool 2: Security Analysis running...
Tool 3: Unit Test Generation running...
Tool 4: Documentation Generation running...

ALL 4 TOOLS COMPLETE!


### View Bug Report

In [17]:
from IPython.display import display, Markdown

display(Markdown("---\n## 🐛 BUG REPORT\n---"))
display(Markdown(bug_result))

---
## 🐛 BUG REPORT
---

Here are the bugs found in the provided Python code:

1. **SQL Injection Vulnerability**
   - Description: The `get_user` function is vulnerable to SQL injection attacks because it directly concatenates user input into the SQL query.
   - Line: 5
   - Severity: CRITICAL
   - Fix: Use parameterized queries instead of string concatenation. Replace the line with: `query = "SELECT * FROM users WHERE username = ?"` and `cursor.execute(query, (username,))`

2. **Plain Text Password Storage**
   - Description: The `get_user` function stores passwords in plain text, which is a significant security risk.
   - Line: 7
   - Severity: CRITICAL
   - Fix: Store hashed versions of passwords instead of plain text. Use a library like `hashlib` or `bcrypt` to hash passwords before storing them.

3. **Division by Zero Error**
   - Description: The `calculate_average` function will raise a `ZeroDivisionError` if the input list `numbers` is empty.
   - Line: 13
   - Severity: HIGH
   - Fix: Add a check to ensure the list is not empty before calculating the average. Replace the return statement with: `return total / len(numbers) if numbers else None`

4. **Hardcoded API Key**
   - Description: The `save_config` function stores a hardcoded API key in a file, which is a security risk.
   - Line: 16
   - Severity: HIGH
   - Fix: Instead of hardcoding the API key, consider using environment variables or a secure secrets management system to store sensitive information.

5. **Lack of Error Handling**
   - Description: The code does not handle potential errors that may occur when connecting to the database or writing to a file.
   - Line: 3, 16
   - Severity: MEDIUM
   - Fix: Add try-except blocks to handle potential errors, such as database connection errors or file I/O errors.

6. **Resource Leak**
   - Description: The `get_user` function does not close the database connection after use.
   - Line: 3
   - Severity: MEDIUM
   - Fix: Add a `finally` block to close the database connection after use, or use a `with` statement to ensure the connection is closed automatically.

Here's an updated version of the code with these bugs fixed:
```python
import sqlite3
import hashlib
import os

def get_user(username, password):
    try:
        with sqlite3.connect('users.db') as conn:
            cursor = conn.cursor()
            query = "SELECT * FROM users WHERE username = ?"
            cursor.execute(query, (username,))
            user = cursor.fetchone()
            if user and user[2] == hashlib.sha256(password.encode()).hexdigest():
                return user
            return None
    except sqlite3.Error as e:
        print(f"Database error: {e}")

def calculate_average(numbers):
    if not numbers:
        return None
    total = sum(numbers)
    return total / len(numbers)

def save_config(api_key):
    try:
        with open('config.txt', 'w') as f:
            f.write(api_key)
    except IOError as e:
        print(f"Error writing to file: {e}")

# Example usage:
api_key = os.environ.get('API_KEY')
save_config(api_key)
```

### View Security Report

In [18]:
display(Markdown("---\n## 🔒 SECURITY REPORT\n---"))
display(Markdown(sec_result))

---
## 🔒 SECURITY REPORT
---

### Security Audit Report

The provided Python code has several security issues that need to be addressed.

#### 1. SQL Injection Vulnerability
* **Name:** SQL Injection
* **Risk:** CRITICAL
* **What attacker can do:** An attacker can inject malicious SQL code to extract or modify sensitive data, potentially leading to unauthorized access or data breaches.
* **How to fix:** Use parameterized queries or prepared statements to separate the SQL code from the user input. In this case, use the `?` placeholder and pass the `username` as a parameter to the `execute()` method:
```python
query  = "SELECT * FROM users WHERE username = ?"
cursor.execute(query, (username,))
```

#### 2. Hardcoded Password Comparison
* **Name:** Hardcoded Password Comparison
* **Risk:** HIGH
* **What attacker can do:** An attacker can obtain the password by reverse-engineering the code or accessing the database, potentially leading to unauthorized access.
* **How to fix:** Store passwords securely using a password hashing algorithm like bcrypt, scrypt, or Argon2. Compare the input password with the stored hash using a secure comparison function:
```python
import bcrypt

# When creating a user
hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())

# When verifying a user
if user and bcrypt.checkpw(password.encode('utf-8'), user[2].encode('utf-8')):
    return user
```

#### 3. Hardcoded API Key
* **Name:** Hardcoded API Key
* **Risk:** HIGH
* **What attacker can do:** An attacker can obtain the API key by accessing the code or the `config.txt` file, potentially leading to unauthorized access to the API.
* **How to fix:** Store the API key securely using environment variables or a secrets management system. Load the API key from the environment variable or secrets manager:
```python
import os

api_key = os.environ.get('API_KEY')
```

#### 4. Missing Input Validation
* **Name:** Missing Input Validation
* **Risk:** MEDIUM
* **What attacker can do:** An attacker can provide malicious input to the `get_user()` function, potentially leading to errors or unexpected behavior.
* **How to fix:** Validate the input `username` and `password` to ensure they meet the expected format and length requirements:
```python
if not isinstance(username, str) or not username.isalnum():
    raise ValueError("Invalid username")
if not isinstance(password, str) or len(password) < 8:
    raise ValueError("Invalid password")
```

#### 5. Missing Error Handling
* **Name:** Missing Error Handling
* **Risk:** MEDIUM
* **What attacker can do:** An attacker can cause the program to crash or produce unexpected errors by providing malicious input.
* **How to fix:** Implement try-except blocks to catch and handle potential errors, such as database connection errors or file I/O errors:
```python
try:
    conn = sqlite3.connect('users.db')
    # ...
except sqlite3.Error as e:
    print(f"Database error: {e}")
```

#### 6. Potential Division by Zero Error
* **Name:** Potential Division by Zero Error
* **Risk:** LOW
* **What attacker can do:** An attacker can cause the program to crash by providing an empty list to the `calculate_average()` function.
* **How to fix:** Add a check to ensure the input list is not empty before calculating the average:
```python
def calculate_average(numbers):
    if not numbers:
        return 0
    total = sum(numbers)
    return total / len(numbers)
```

### View Generated Unit Tests

In [19]:
display(Markdown("---\n## 🧪 TEST REPORT\n---"))
display(Markdown(test_result))

---
## 🧪 TEST REPORT
---

Here's how you can write unit tests for the given Python code using pytest. 

```python
# tests/test_functions.py
import pytest
import sqlite3
import os
from your_module import get_user, calculate_average, save_config  # Replace 'your_module' with the actual name of your module

# Create a test database for the get_user function
def create_test_database():
    conn = sqlite3.connect('test_users.db')
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, username TEXT, password TEXT)")
    conn.commit()
    conn.close()

# Delete the test database after the tests are done
def delete_test_database():
    os.remove('test_users.db')

# Create a test configuration file for the save_config function
def create_test_config_file():
    with open('test_config.txt', 'w') as f:
        f.write("")

# Delete the test configuration file after the tests are done
def delete_test_config_file():
    os.remove('test_config.txt')

# Test the get_user function
@pytest.fixture
def setup_get_user():
    create_test_database()
    conn = sqlite3.connect('test_users.db')
    cursor = conn.cursor()
    cursor.execute("INSERT INTO users (username, password) VALUES ('test_user', 'test_password')")
    conn.commit()
    conn.close()
    yield
    delete_test_database()

def test_get_user_normal_case(setup_get_user):
    user = get_user('test_user', 'test_user.db')
    assert user is None  # The database name is incorrect, so it should return None

    # Use the correct database name
    conn = sqlite3.connect('test_users.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE username = 'test_user'")
    user = cursor.fetchone()
    conn.close()

    user = get_user('test_user', 'test_users.db')
    assert user is None  # The password is incorrect, so it should return None

    user = get_user('test_user', 'test_password')
    assert user is not None  # The username and password are correct, so it should return the user

def test_get_user_edge_case():
    user = get_user('', 'test_password')
    assert user is None  # The username is empty, so it should return None

def test_get_user_error_case():
    with pytest.raises(sqlite3.OperationalError):
        get_user('test_user', 'non_existent_database.db')

# Test the calculate_average function
def test_calculate_average_normal_case():
    numbers = [1, 2, 3, 4, 5]
    average = calculate_average(numbers)
    assert average == 3  # The average of the numbers is 3

def test_calculate_average_edge_case():
    numbers = [0, 0, 0, 0, 0]
    average = calculate_average(numbers)
    assert average == 0  # The average of the numbers is 0

def test_calculate_average_error_case():
    numbers = []
    with pytest.raises(ZeroDivisionError):
        calculate_average(numbers)  # The list is empty, so it should raise a ZeroDivisionError

# Test the save_config function
def test_save_config_normal_case(tmp_path):
    save_config()
    assert os.path.exists('config.txt')  # The configuration file should exist
    with open('config.txt', 'r') as f:
        api_key = f.read()
    assert api_key == "sk-secret123456789"  # The API key should be correct

def test_save_config_error_case(tmp_path):
    # Try to save the configuration file to a directory that does not exist
    with pytest.raises(FileNotFoundError):
        with open('non_existent_directory/config.txt', 'w') as f:
            f.write("sk-secret123456789")

# Create a test configuration file for the save_config function
def test_save_config_file_exists():
    create_test_config_file()
    save_config()
    assert os.path.exists('config.txt')  # The configuration file should exist
    delete_test_config_file()
```

Remember to replace `'your_module'` with the actual name of your module.

Also, make sure to run these tests in a separate environment to avoid overwriting any existing configuration files or databases.

To run these tests, save them in a file named `test_functions.py` and run the command `pytest` in the terminal. 

Please note that these tests are just examples and you may need to adjust them according to your specific use case. 

Also, the `get_user` function is vulnerable to SQL injection attacks. You should use parameterized queries or an ORM to prevent this. 

The `calculate_average` function does not handle the case where the input list is empty. You should add a check for this and raise a meaningful error or return a special value. 

The `save_config` function overwrites any existing configuration file. You should add a check for this and raise a meaningful error or return a special value. 

The `save_config` function uses a hardcoded API key. You should consider using environment variables or a secure storage mechanism to store sensitive data.

## View Generated Documentation

In [20]:
display(Markdown("---\n## 📄 DOCUMENTATION\n---"))
display(Markdown(docs_result))

---
## 📄 DOCUMENTATION
---

**User Management and Utility Functions**
=====================================

### Overview

This module provides a set of functions for user management and utility purposes. It includes functions to retrieve user information from a SQLite database, calculate the average of a list of numbers, and save a configuration file.

### Functions

#### 1. `get_user(username, password)`
--------------------------------

*   **Parameters:**
    *   `username` (str): The username to search for in the database.
    *   `password` (str): The password to verify for the given username.
*   **Returns:**
    *   `tuple` or `None`: A tuple containing the user's information if the username and password match, otherwise `None`.
*   **Example:**
    ```python
user = get_user('john_doe', 'password123')
if user:
    print(user)  # Output: (1, 'john_doe', 'password123', ...)
else:
    print("User not found or incorrect password")
```

#### 2. `calculate_average(numbers)`
---------------------------------

*   **Parameters:**
    *   `numbers` (list): A list of numbers to calculate the average of.
*   **Returns:**
    *   `float`: The average of the given numbers.
*   **Example:**
    ```python
numbers = [1, 2, 3, 4, 5]
average = calculate_average(numbers)
print(average)  # Output: 3.0
```

#### 3. `save_config()`
---------------------

*   **Parameters:** None
*   **Returns:** None
*   **Example:**
    ```python
save_config()  # Saves the API key to a file named 'config.txt'
```

### Dependencies

*   `sqlite3`: A built-in Python library for interacting with SQLite databases.

### Usage Example

```python
import sqlite3

# Create a sample database
conn = sqlite3.connect('users.db')
cursor = conn.cursor()
cursor.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, username TEXT, password TEXT)')
cursor.execute('INSERT INTO users (username, password) VALUES (?, ?)', ('john_doe', 'password123'))
conn.commit()
conn.close()

# Get a user
user = get_user('john_doe', 'password123')
if user:
    print(user)  # Output: (1, 'john_doe', 'password123', ...)

# Calculate the average of a list of numbers
numbers = [1, 2, 3, 4, 5]
average = calculate_average(numbers)
print(average)  # Output: 3.0

# Save the configuration
save_config()  # Saves the API key to a file named 'config.txt'
```

**Security Note:** The `get_user` function is vulnerable to SQL injection attacks. In a real-world application, consider using parameterized queries or an ORM to prevent such vulnerabilities. Additionally, storing passwords in plain text is not recommended; consider using a secure password hashing library instead.

In [21]:

from IPython.display import display, Markdown

my_code = """
def divide(a, b):
    return a / b

def find_max(items):
    max_val = items[0]
    for item in items:
        if item > max_val:
            max_val = item
    return max_val
"""

my_language = "Python"

result = detect_bugs(my_code, my_language)
display(Markdown(result))

Tool 1: Bug Detection running...


Here are the bugs found in the code:

1. **Division by zero error**
   - Description: The `divide` function does not handle division by zero, which will raise a `ZeroDivisionError`.
   - Line: 2
   - Severity: CRITICAL
   - Fix: Add a check to ensure that `b` is not zero before performing the division. 
     ```python
def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b
```

2. **Index out of range error**
   - Description: The `find_max` function assumes that the input list `items` is not empty and tries to access the first element (`items[0]`). If the list is empty, this will raise an `IndexError`.
   - Line: 4
   - Severity: CRITICAL
   - Fix: Add a check to ensure that the list is not empty before trying to access its elements.
     ```python
def find_max(items):
    if not items:
        raise ValueError("List is empty")
    max_val = items[0]
    for item in items:
        if item > max_val:
            max_val = item
    return max_val
```

3. **Type error**
   - Description: The `find_max` function does not check the type of the input `items`. If `items` is not a list or if the list contains non-comparable elements, this will raise a `TypeError`.
   - Line: 4
   - Severity: HIGH
   - Fix: Add a check to ensure that `items` is a list and that all elements in the list are comparable.
     ```python
def find_max(items):
    if not isinstance(items, list):
        raise TypeError("Input must be a list")
    if not items:
        raise ValueError("List is empty")
    try:
        max_val = items[0]
        for item in items:
            if item > max_val:
                max_val = item
        return max_val
    except TypeError:
        raise TypeError("List contains non-comparable elements")
```

4. **Inefficient algorithm**
   - Description: The `find_max` function has a time complexity of O(n) because it iterates over the list to find the maximum value. However, Python has a built-in `max` function that can do this more efficiently.
   - Line: 4-7
   - Severity: LOW
   - Fix: Use the built-in `max` function to find the maximum value in the list.
     ```python
def find_max(items):
    if not items:
        raise ValueError("List is empty")
    return max(items)
```

In [ ]:
import os

groq_key = os.environ.get('GROQ_API_KEY', '')

app_code = '''import streamlit as st
import os, time
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"] = "''' + groq_key + '''"

def get_llm():
    return ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"], model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=2048)

def run_tool(sys_msg, human_msg, code, language):
    prompt = ChatPromptTemplate.from_messages([("system", sys_msg), ("human", human_msg)])
    return (prompt | get_llm() | StrOutputParser()).invoke({"language": language, "code": code})

st.set_page_config(page_title="AI Code Review Agent", layout="centered")
st.title("AI Code Review Agent")

language = st.selectbox("Language", ["Python", "JavaScript", "Java", "TypeScript", "C++"])
code     = st.text_area("Paste your code:", height=250)

col1, col2, col3, col4 = st.columns(4)
do_bugs = col1.checkbox("Bugs",     value=True)
do_sec  = col2.checkbox("Security", value=True)
do_test = col3.checkbox("Tests",    value=True)
do_docs = col4.checkbox("Docs",     value=True)

if st.button("Review Code", type="primary", use_container_width=True):
    if not code.strip():
        st.error("Paste some code first!")
        st.stop()

    results = {}
    tools   = sum([do_bugs, do_sec, do_test, do_docs])
    done    = 0
    bar     = st.progress(0)
    status  = st.empty()
    start   = time.time()

    if do_bugs:
        status.text("Running Bug Detection...")
        results["Bugs"]     = run_tool("Find ALL bugs. For each: number, description, severity, fix.", "Find bugs in {language}:\\n{code}", code, language)
        done += 1; bar.progress(done / tools)

    if do_sec:
        status.text("Running Security Analysis...")
        results["Security"] = run_tool("Find ALL security issues. For each: name, risk, impact, fix.", "Security audit {language}:\\n{code}", code, language)
        done += 1; bar.progress(done / tools)

    if do_test:
        status.text("Generating Unit Tests...")
        results["Tests"]    = run_tool("Write complete pytest unit tests. Cover normal, edge, error cases.", "Write tests for {language}:\\n{code}", code, language)
        done += 1; bar.progress(done / tools)

    if do_docs:
        status.text("Generating Documentation...")
        results["Docs"]     = run_tool("Write professional docs: overview, functions, params, returns, usage.", "Document {language}:\\n{code}", code, language)
        done += 1; bar.progress(done / tools)

    elapsed = round(time.time() - start, 1)
    status.text(f"Done in {elapsed}s!")

    for tab, (name, content) in zip(st.tabs(list(results.keys())), results.items()):
        with tab:
            st.markdown(content)
            st.download_button("Download", content, f"{name}_report.md", key=name)
'''

with open("agent_app.py", "w") as f:
    f.write(app_code)

print("agent_app.py created!")
print(f"Run: streamlit run agent_app.py")

agent_app.py created!
Run: streamlit run agent_app.py
